# 🤝 Notebook 2 — Hinted Handoff: Don't Drop the Write, *Hold* It

## 💡 The idea in one sentence

> When the coordinator can't reach a replica, it **writes itself a sticky note** (a *hint*): 
> _'Next time `r3` is back, hand it this write.'_ When `r3` recovers, the coordinator **drains the sticky notes**.

This is the same trick your mail carrier uses when a mailbox is blocked: they keep the letters with them and re-deliver when access is restored.

Cassandra, Amazon DynamoDB (original Dynamo paper), Riak and ScyllaDB all implement this pattern to stay **highly available** for writes without giving up eventual convergence.

## 🛠️ Setup

```bash
cd 02-distributed-primitives/hinted-handoff
uv sync
```

Pick the `.venv` kernel in VS Code (top-right). If it doesn't appear, `Cmd+Shift+P` → **Reload Window**.

## 🎯 What you'll build here

A `Coordinator` that:
1. Tries to forward every write to every replica.
2. On a failed replica, **stores a hint** `(target, key, value, timestamp)` in a local queue.
3. On recovery, **replays the hints** to the replica in **timestamp order** so later writes overwrite earlier ones correctly.

We'll compare the final state with the broken version from Notebook 1.

## 🟩 GOOD: coordinator with a hint log

In [ ]:
import time
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import Deque, Dict, List, Tuple

@dataclass
class Replica:
    """Storage node that remembers the timestamp of the latest write per key.

    Using a timestamp on every stored value is what lets us replay hints safely:
    if a newer write already landed while the node was partially online,
    an older replayed hint won't clobber it.
    """
    name: str
    up: bool = True
    data: Dict[str, Tuple[str, int]] = field(default_factory=dict)  # key -> (value, ts)

    def write(self, key: str, value: str, ts: int) -> bool:
        if not self.up:
            return False
        cur = self.data.get(key)
        # 'Last-write-wins' using the coordinator's timestamp.
        if cur is None or ts >= cur[1]:
            self.data[key] = (value, ts)
        return True


@dataclass
class Hint:
    target: str   # which replica the hint is for
    key: str
    value: str
    ts: int       # logical timestamp assigned by the coordinator


class Coordinator:
    """Fan-out writer that stores hints for unreachable replicas."""

    def __init__(self, replicas: List[Replica]) -> None:
        self.replicas = replicas
        # One FIFO queue of hints per target replica.
        self.hints: Dict[str, Deque[Hint]] = defaultdict(deque)
        self._clock = 0  # monotonic logical clock

    def _next_ts(self) -> int:
        self._clock += 1
        return self._clock

    def write(self, key: str, value: str) -> None:
        ts = self._next_ts()
        print(f'coordinator: write {key}={value} @ ts={ts}')
        for r in self.replicas:
            if r.write(key, value, ts):
                print(f'  ✅ {r.name} accepted')
            else:
                # Replica is down — remember the write for later.
                self.hints[r.name].append(Hint(r.name, key, value, ts))
                print(f'  📝 stored hint for {r.name}: {key}={value} @ ts={ts}')

    def deliver_hints(self) -> None:
        """Replay queued hints for any replica that's back online."""
        for r in self.replicas:
            if not r.up or not self.hints[r.name]:
                continue
            delivered = 0
            while self.hints[r.name]:
                h = self.hints[r.name].popleft()
                r.write(h.key, h.value, h.ts)
                delivered += 1
            print(f'  ✉ delivered {delivered} hint(s) to {r.name}')


### ▶️ Run the scenario

In [ ]:
replicas = [Replica('r1'), Replica('r2'), Replica('r3')]
coord = Coordinator(replicas)

print('--- r3 goes DOWN; 5 writes arrive ---')
replicas[2].up = False
for i in range(5):
    coord.write(f'k{i}', f'v{i}')

print(f'\nhints pending for r3: {list(coord.hints["r3"])}')

print('\n--- r3 recovers; coordinator drains hints ---')
replicas[2].up = True
coord.deliver_hints()

print('\nFinal state:')
for r in replicas:
    print(f'  {r.name}: {r.data}')

# Convergence is the claim, so check it: all three replicas hold identical data,
# with identical timestamps, and no hint is left behind.
values = [{k: v for k, v in r.data.items()} for r in replicas]
assert values[0] == values[1] == values[2], values
assert len(values[0]) == 5
assert not any(coord.hints[r.name] for r in replicas), 'hints were not fully drained'
print('\n✔ all three replicas converged, and the hint queue is empty')


### ✅ What changed vs Notebook 1

- Every write that targeted `r3` while it was down is buffered as a **hint** on the coordinator.
- When `r3` comes back, the coordinator replays those hints *in order*, with their original timestamps.
- All three replicas converge — **without** running a full background repair scan.

## 🧪 Edge case: a newer write arrives *before* the hint is replayed

Suppose `r3` recovers, a client writes `k2=v2-fresh`, and *only then* the coordinator starts replaying hints. 
The old hint for `k2=v2` must **not** overwrite the newer value. This is exactly what the timestamp check in `Replica.write` protects us from.

In [ ]:
replicas = [Replica('r1'), Replica('r2'), Replica('r3')]
coord = Coordinator(replicas)

replicas[2].up = False
coord.write('k2', 'v2-old')    # queued as hint for r3
replicas[2].up = True

# A newer write lands on r3 directly, before hints are drained.
coord.write('k2', 'v2-fresh')
print('r3 before replay:', replicas[2].data)

coord.deliver_hints()
print('r3 after replay :', replicas[2].data)
assert replicas[2].data['k2'][0] == 'v2-fresh', 'Stale hint should NOT overwrite newer write'
print('✅ last-write-wins held — the stale hint was ignored')


## 🗺️ Mental model

```
           ┌──────────────┐                
writes ──▶ │  Coordinator │ ──▶ r1 ✅      
           │   + hint log │ ──▶ r2 ✅      
           └─────┬────────┘ ──▶ r3 ✗ (down) 
                 │  📝 hint(r3, k=v, ts)    
                 ▼                          
        r3 recovers → drain queue ➜ r3 ✅  
```

Hints are **per-target, FIFO, and timestamped**. 
That's the whole trick.

👉 But hints aren't magic. What if the coordinator itself crashes? 
What if `r3` stays down for *days*? 
Notebook 3 covers the limits — TTLs, sloppy quorum, and when you still need anti-entropy repair.